In [ ]:
IS_COLAB = False
# IS_COLAB = True

In [ ]:
# Paths for local and Colab environments

PYTHON_SCRIPT_LOCAL = "main.py" # local
PYTHON_SCRIPT_COLAB = "/content/drive/MyDrive/mmdit-motion/main.py" # colab

DATA_ROOT_LOCAL = "data/HumanML3D" # local
DATA_ROOT_COLAB = "/content/drive/MyDrive/mmdit-motion/data/HumanML3D" # colab

CKPT_ROOT_LOCAL = "models/ckpt.pt" # local
CKPT_ROOT_COLAB = "/content/drive/MyDrive/mmdit-motion/models/ckpt.pt" # colab

GRADIO_DEMO_SCRIPT_LOCAL = "gradio-test.py" # local
GRADIO_DEMO_SCRIPT_COLAB = "/content/drive/MyDrive/mmdit-motion/gradio-test.py" # colab

if IS_COLAB:
    PYTHON_SCRIPT = PYTHON_SCRIPT_COLAB
    DATA_ROOT = DATA_ROOT_COLAB
    CKPT_ROOT = CKPT_ROOT_COLAB
    GRADIO_DEMO_SCRIPT = GRADIO_DEMO_SCRIPT_COLAB
else:
    PYTHON_SCRIPT = PYTHON_SCRIPT_LOCAL
    DATA_ROOT = DATA_ROOT_LOCAL
    CKPT_ROOT = CKPT_ROOT_LOCAL
    GRADIO_DEMO_SCRIPT = GRADIO_DEMO_SCRIPT_LOCAL

In [ ]:
# If using Colab, mount Google Drive and install dependencies

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    !pip install transformers mmdit torchinfo gradio

    !nvidia-smi --query-gpu=name --format=csv,noheader


In [ ]:
# Summary model info

!python $PYTHON_SCRIPT summary --dim_motion 512 --depth 8

In [ ]:
# Train the model for 1 step (for testing)

!python $PYTHON_SCRIPT train \
    --text_encoder dummy \
    --dim_motion 512 --depth 8 \
    --batch_size 64 \
    --steps 1 \
    --save_every 1 \
    --ckpt_out $CKPT_ROOT

In [ ]:
# Sample motion from the trained model (for testing)

!python $PYTHON_SCRIPT sample \
    --resume $CKPT_ROOT \
    --text_encoder dummy \
    --dim_motion 512 --depth 8 \
    --prompt "a person walks forward then sits down" \
    --steps 50 --cfg 4.0 \
    --out sample.npy

In [ ]:
# Train the model

!python $PYTHON_SCRIPT train \
    --data_root $DATA_ROOT \
    --text_encoder clip \
    --dim_motion 512 --depth 8 \
    --batch_size 64 \
    --steps 200000 \
    --ckpt_out $CKPT_ROOT

# Terminate Colab runtime to free up resources after training

if IS_COLAB:
    from google.colab import runtime
    runtime.unassign()

In [ ]:
# Train the model on H100
# bf16 + flash + tf32 + torch.compile
# save every 5000 steps

!python $PYTHON_SCRIPT train \
    --data_root $DATA_ROOT \
    --text_encoder clip \
    --dim_motion 512 --depth 8 \
    --batch_size 64 \
    --steps 200000 \
    --amp_dtype bf16 \
    --compile \
    --save_every 5000 \
    --ckpt_out $CKPT_ROOT

# Terminate Colab runtime to free up resources after training

if IS_COLAB:
    from google.colab import runtime
    runtime.unassign()

In [ ]:
# Resume training from a checkpoint on H100

!python $PYTHON_SCRIPT train \
    --data_root $DATA_ROOT \
    --text_encoder clip \
    --dim_motion 512 --depth 8 \
    --batch_size 64 \
    --steps 400000 \
    --amp_dtype bf16 \
    --compile \
    --resume $CKPT_ROOT \
    --save_every 5000 \
    --ckpt_out $CKPT_ROOT

# Terminate Colab runtime to free up resources after training

if IS_COLAB:
    from google.colab import runtime
    runtime.unassign()

In [ ]:
# Sample motion from the trained model

!python $PYTHON_SCRIPT sample \
    --resume $CKPT_ROOT \
    --text_encoder clip \
    --dim_motion 512 --depth 8 \
    --prompt "a person walks forward then sits down" \
    --steps 50 --cfg 4.0 \
    --out sample.npy

In [ ]:
# Run the Gradio demo

SHARE = "--share" if IS_COLAB else ""

!python $GRADIO_DEMO_SCRIPT \
    --ckpt $CKPT_ROOT \
    --use_ema \
    $SHARE